<a href="https://colab.research.google.com/github/WhySoSerious404/trust-safety-analytics/blob/main/02-abuse-detection-sql/abuse_detection_sql.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Abuse detection in SQL

Generate synthetic platform activity, detect planted patterns
using SQL, and evaluate account-level flags.

In [1]:
from google.colab import drive
from pathlib import Path

drive.mount('/content/drive')

DATA_DIR = Path(
    "/content/drive/MyDrive/Project: trust-safety/"
    "02-abuse-detection-sql/data"
)

DATA_DIR.mkdir(parents=True, exist_ok=True)
print('Data folder:', DATA_DIR)

Mounted at /content/drive
Data folder: /content/drive/MyDrive/Project: trust-safety/02-abuse-detection-sql/data


## Synthetic data generator

In [2]:
import numpy as np
import pandas as pd

SEED = 42
rng = np.random.default_rng(SEED)

N_ACCOUNTS = 20000
TARGET_EVENTS = 300000

START = pd.Timestamp("2026-09-01", tz="UTC")
END = START + pd.Timedelta(days=30)

PATTERN_COUNTS = {
    "legitimate": 19000,
    "spam": 400,
    "takeover": 300,
    "automated_clicking": 300
}

assert sum(PATTERN_COUNTS.values()) == N_ACCOUNTS

print("Accounts:", N_ACCOUNTS)
print("Target events:", TARGET_EVENTS)
print("Observation period:", START, "to", END, "(end excluded)")

pd.Series(PATTERN_COUNTS, name="account_count")

Accounts: 20000
Target events: 300000
Observation period: 2026-09-01 00:00:00+00:00 to 2026-10-01 00:00:00+00:00 (end excluded)


,account_count
legitimate,19000
spam,400
takeover,300
automated_clicking,300


In [3]:
account_ids = np.arange(1, N_ACCOUNTS + 1)

signup_dates = START - pd.to_timedelta(
    rng.integers(1, 61, size=N_ACCOUNTS),
    unit="D"
)

accounts = pd.DataFrame({
    "account_id": account_ids,
    "signup_date": signup_dates.strftime("%Y-%m-%d"),
    "country": rng.choice(
        ["IN", "US", "GB", "DE", "CA"],
        size=N_ACCOUNTS
    )
})

labels = np.repeat(
    list(PATTERN_COUNTS.keys()),
    list(PATTERN_COUNTS.values())
)

rng.shuffle(labels)

ground_truth = pd.DataFrame({
    "account_id": account_ids,
    "pattern": labels
})

assert accounts["account_id"].is_unique
assert len(accounts) == N_ACCOUNTS

display(accounts.head())
ground_truth["pattern"].value_counts()

,account_id,signup_date,country
0,1,2026-08-26,IN
1,2,2026-07-16,DE
2,3,2026-07-23,US
3,4,2026-08-05,DE
4,5,2026-08-06,GB


,count
pattern,
legitimate,19000
spam,400
takeover,300
automated_clicking,300


In [4]:
base_account_ids = np.repeat(
    accounts["account_id"].to_numpy(),
    10
)

extra_count = TARGET_EVENTS - len(base_account_ids)

extra_account_ids = rng.choice(
    accounts["account_id"].to_numpy(),
    size=extra_count
)

event_account_ids = np.concatenate([
    base_account_ids,
    extra_account_ids
])

seconds_in_period = int((END - START).total_seconds())

event_offsets = rng.integers(
    0,
    seconds_in_period,
    size=TARGET_EVENTS
)

events = pd.DataFrame({
    "account_id": event_account_ids,
    "event_time": START + pd.to_timedelta(event_offsets, unit="s"),
    "event_type": rng.choice(
        ["login", "post", "click"],
        size=TARGET_EVENTS,
        p=[0.3, 0.2, 0.5]
    )
})

country_lookup = accounts.set_index("account_id")["country"]
events["country"] = events["account_id"].map(country_lookup)

print("Baseline events:", len(events))
print("Minimum events per account:",
      events.groupby("account_id").size().min())

events.head()

Baseline events: 300000
Minimum events per account: 10


,account_id,event_time,event_type,country
0,1,2026-09-23 17:53:31+00:00,click,IN
1,1,2026-09-28 12:05:56+00:00,click,IN
2,1,2026-09-02 17:53:48+00:00,login,IN
3,1,2026-09-09 14:44:31+00:00,post,IN
4,1,2026-09-30 07:48:46+00:00,login,IN


In [5]:
ip_lookup = {}

for account_id in accounts["account_id"]:
    a, b, c = rng.integers(1, 255, size=3)
    ip_lookup[account_id] = f"10.{a}.{b}.{c}"

events["ip_address"] = events["account_id"].map(ip_lookup)

assert events["ip_address"].notna().all()

events.head()

,account_id,event_time,event_type,country,ip_address
0,1,2026-09-23 17:53:31+00:00,click,IN,10.145.131.65
1,1,2026-09-28 12:05:56+00:00,click,IN,10.145.131.65
2,1,2026-09-02 17:53:48+00:00,login,IN,10.145.131.65
3,1,2026-09-09 14:44:31+00:00,post,IN,10.145.131.65
4,1,2026-09-30 07:48:46+00:00,login,IN,10.145.131.65


## Constructing the simulation

### Spam IDs

In [6]:
spam_ids = ground_truth.loc[
    ground_truth["pattern"] == "spam",
    "account_id"
].to_numpy()

rng.shuffle(spam_ids)

spam_gangs = np.array_split(spam_ids, 20)

print("Planted spam accounts:", len(spam_ids))
print("Number of gangs:", len(spam_gangs))
print("Accounts per gang:", len(spam_gangs[0]))
print("First gang:", spam_gangs[0])

Planted spam accounts: 400
Number of gangs: 20
Accounts per gang: 20
First gang: [  339   202  9086  1596 10360 16164  5010 18614  5333  7844  8127 12664
  1291 12808  3930 10754  4544 15316  2621  9276]


In [7]:
account_event_rows = events.groupby("account_id").groups

changed_spam_rows = []

for gang_number, gang in enumerate(spam_gangs, start=1):
    start_offset = int(
        rng.integers(0, seconds_in_period - 600)
    )

    gang_start = START + pd.Timedelta(seconds=start_offset)

    for account_id in gang:
        account_rows = account_event_rows[account_id]

        max_posts = min(12, len(account_rows))
        n_posts = int(rng.integers(2, max_posts + 1))

        selected_rows = rng.choice(
            account_rows,
            size=n_posts,
            replace=False
        )

        events.loc[selected_rows, "event_type"] = "post"

        events.loc[selected_rows, "event_time"] = (
            gang_start
            + pd.to_timedelta(
                rng.integers(0, 600, size=n_posts),
                unit="s"
            )
        )

        last_octet = int(rng.integers(1, 255))

        events.loc[selected_rows, "ip_address"] = (
            f"10.250.{gang_number}.{last_octet}"
        )

        changed_spam_rows.extend(selected_rows.tolist())

assert len(events) == TARGET_EVENTS

print("Events modified:", len(changed_spam_rows))
print("Total events:", len(events))

display(events.loc[changed_spam_rows[:5]])

Events modified: 2885
Total events: 300000


,account_id,event_time,event_type,country,ip_address
3380,339,2026-09-27 15:33:16+00:00,post,IN,10.250.1.34
239094,339,2026-09-27 15:41:52+00:00,post,IN,10.250.1.34
3385,339,2026-09-27 15:34:12+00:00,post,IN,10.250.1.34
3386,339,2026-09-27 15:41:45+00:00,post,IN,10.250.1.34
3382,339,2026-09-27 15:33:44+00:00,post,IN,10.250.1.34


In [8]:
spam_check = events.loc[changed_spam_rows]

assert spam_check.index.is_unique
assert set(spam_check["account_id"]) == set(spam_ids)
assert spam_check["event_type"].eq("post").all()

gang_checks = []

for gang_number, gang in enumerate(spam_gangs, start=1):
    gang_posts = spam_check[
        spam_check["account_id"].isin(gang)
    ]

    span_seconds = (
        gang_posts["event_time"].max()
        - gang_posts["event_time"].min()
    ).total_seconds()

    ip_blocks = (
        gang_posts["ip_address"]
        .str.rsplit(".", n=1)
        .str[0]
    )

    assert span_seconds < 600
    assert ip_blocks.nunique() == 1

    gang_checks.append({
        "gang": gang_number,
        "accounts": gang_posts["account_id"].nunique(),
        "posts": len(gang_posts),
        "span_seconds": span_seconds,
        "ip_block": ip_blocks.iloc[0]
    })

print("Spam accounts represented:", spam_check["account_id"].nunique())
print("Gangs checked:", len(gang_checks))

pd.DataFrame(gang_checks).head()

Spam accounts represented: 400
Gangs checked: 20


,gang,accounts,posts,span_seconds,ip_block
0,1,20,155,594.0,10.250.1
1,2,20,133,597.0,10.250.2
2,3,20,138,593.0,10.250.3
3,4,20,150,599.0,10.250.4
4,5,20,148,592.0,10.250.5


In [9]:
spam_check = events.loc[changed_spam_rows]

assert spam_check.index.is_unique
assert set(spam_check["account_id"]) == set(spam_ids)
assert spam_check["event_type"].eq("post").all()

gang_checks = []

for gang_number, gang in enumerate(spam_gangs, start=1):
    gang_posts = spam_check[
        spam_check["account_id"].isin(gang)
    ]

    span_seconds = (
        gang_posts["event_time"].max()
        - gang_posts["event_time"].min()
    ).total_seconds()

    ip_blocks = (
        gang_posts["ip_address"]
        .str.rsplit(".", n=1)
        .str[0]
    )

    assert span_seconds < 600
    assert ip_blocks.nunique() == 1

    gang_checks.append({
        "gang": gang_number,
        "accounts": gang_posts["account_id"].nunique(),
        "posts": len(gang_posts),
        "span_seconds": span_seconds,
        "ip_block": ip_blocks.iloc[0]
    })

print("Spam accounts represented:", spam_check["account_id"].nunique())
print("Gangs checked:", len(gang_checks))

pd.DataFrame(gang_checks).head()

Spam accounts represented: 400
Gangs checked: 20


,gang,accounts,posts,span_seconds,ip_block
0,1,20,155,594.0,10.250.1
1,2,20,133,597.0,10.250.2
2,3,20,138,593.0,10.250.3
3,4,20,150,599.0,10.250.4
4,5,20,148,592.0,10.250.5


### Takeover IDs

In [10]:
takeover_ids = ground_truth.loc[
    ground_truth["pattern"] == "takeover",
    "account_id"
].to_numpy()

event_counts = events.groupby("account_id").size()

print("Planted takeover accounts:", len(takeover_ids))
print(
    "Minimum available events:",
    event_counts.loc[takeover_ids].min()
)

Planted takeover accounts: 300
Minimum available events: 10


In [12]:
countries = accounts["country"].unique()

takeover_plan_rows = []

for account_id in takeover_ids:
    home_country = country_lookup.loc[account_id]

    other_countries = [
        country
        for country in countries
        if country != home_country
    ]

    start_offset = int(
        rng.integers(
            7 * 24 * 60 * 60,
            seconds_in_period - 3600
        )
    )

    takeover_plan_rows.append({
        "account_id": account_id,
        "home_country": home_country,
        "new_country": rng.choice(other_countries),
        "takeover_time": START + pd.Timedelta(
            seconds=start_offset
        ),
        "burst_seconds": int(rng.choice([600, 1800, 3600]))
    })

takeover_plan = pd.DataFrame(takeover_plan_rows)

assert takeover_plan["home_country"].ne(
    takeover_plan["new_country"]
).all()

print("Takeover plans:", len(takeover_plan))
takeover_plan.head()

Takeover plans: 300


,account_id,home_country,new_country,takeover_time,burst_seconds
0,31,CA,GB,2026-09-19 04:33:52+00:00,1800
1,34,DE,US,2026-09-22 15:38:57+00:00,3600
2,41,DE,CA,2026-09-25 03:31:22+00:00,1800
3,103,DE,GB,2026-09-09 21:54:01+00:00,1800
4,132,CA,DE,2026-09-25 15:48:38+00:00,1800


In [13]:
changed_takeover_rows = []
takeover_history_rows = []

for _, plan in takeover_plan.iterrows():
    account_id = plan["account_id"]

    account_rows = rng.permutation(
        account_event_rows[account_id]
    )

    history_rows = account_rows[:6]
    attack_rows = account_rows[6:]

    events.loc[history_rows, "event_time"] = (
        START
        + pd.to_timedelta(
            rng.integers(0, 7 * 24 * 60 * 60, size=6),
            unit="s"
        )
    )

    events.loc[history_rows, "country"] = plan["home_country"]
    events.loc[history_rows, "ip_address"] = ip_lookup[account_id]
    events.loc[history_rows[0], "event_type"] = "login"

    attack_offsets = rng.integers(
        1,
        int(plan["burst_seconds"]),
        size=len(attack_rows)
    )
    attack_offsets[0] = 0

    events.loc[attack_rows, "event_time"] = (
        plan["takeover_time"]
        + pd.to_timedelta(attack_offsets, unit="s")
    )

    events.loc[attack_rows, "country"] = plan["new_country"]

    events.loc[attack_rows, "event_type"] = rng.choice(
        ["post", "click"],
        size=len(attack_rows)
    )
    events.loc[attack_rows[0], "event_type"] = "login"

    a, b, c = rng.integers(1, 255, size=3)
    events.loc[attack_rows, "ip_address"] = f"10.{a}.{b}.{c}"

    takeover_history_rows.extend(history_rows.tolist())
    changed_takeover_rows.extend(attack_rows.tolist())

assert len(events) == TARGET_EVENTS

print("History events:", len(takeover_history_rows))
print("Takeover accounts:",
      events.loc[changed_takeover_rows, "account_id"].nunique())
print("Total events:", len(events))

example_id = takeover_ids[0]

display(
    events.loc[events["account_id"] == example_id]
    .sort_values("event_time")
)

History events: 1800
Takeover accounts: 300
Total events: 300000


,account_id,event_time,event_type,country,ip_address
277911,31,2026-09-02 01:25:00+00:00,post,CA,10.208.75.175
309,31,2026-09-02 05:57:11+00:00,click,CA,10.208.75.175
304,31,2026-09-02 16:52:09+00:00,post,CA,10.208.75.175
301,31,2026-09-06 00:25:14+00:00,login,CA,10.208.75.175
303,31,2026-09-07 14:36:59+00:00,click,CA,10.208.75.175
295427,31,2026-09-07 19:35:06+00:00,click,CA,10.208.75.175
300,31,2026-09-19 04:33:52+00:00,login,GB,10.164.109.104
307,31,2026-09-19 04:39:14+00:00,post,GB,10.164.109.104
259841,31,2026-09-19 04:39:54+00:00,post,GB,10.164.109.104
308,31,2026-09-19 04:39:56+00:00,click,GB,10.164.109.104


In [14]:
for ri, plan in takeover_plan.iterrows():
    activity = events.loc[
        account_event_rows[plan["account_id"]]
    ]

    history = activity[
        activity["event_time"] < plan["takeover_time"]
    ]

    attack = activity[
        activity["event_time"] >= plan["takeover_time"]
    ]

    assert len(history) == 6
    assert history["country"].eq(plan["home_country"]).all()
    assert history["event_type"].eq("login").any()

    assert attack["country"].eq(plan["new_country"]).all()

    burst_end = plan["takeover_time"] + pd.Timedelta(
        seconds=int(plan["burst_seconds"])
    )

    assert attack["event_time"].lt(burst_end).all()

    first_attack = attack.sort_values("event_time").iloc[0]

    assert first_attack["event_time"] == plan["takeover_time"]
    assert first_attack["event_type"] == "login"

print("Verified takeover accounts:", len(takeover_plan))

Verified takeover accounts: 300


### Click IDs

In [15]:
click_ids = ground_truth.loc[
    ground_truth["pattern"] == "automated_clicking",
    "account_id"
].to_numpy()

changed_click_rows = []
click_plan_rows = []

for account_id in click_ids:
    account_rows = account_event_rows[account_id]

    max_clicks = min(14, len(account_rows))
    n_clicks = int(rng.integers(4, max_clicks + 1))

    burst_seconds = int(rng.choice([10, 30, 120]))

    start_offset = int(
        rng.integers(0, seconds_in_period - burst_seconds)
    )
    burst_start = START + pd.Timedelta(seconds=start_offset)

    selected_rows = rng.choice(
        account_rows,
        size=n_clicks,
        replace=False
    )

    events.loc[selected_rows, "event_type"] = "click"

    events.loc[selected_rows, "event_time"] = (
        burst_start
        + pd.to_timedelta(
            rng.integers(0, burst_seconds, size=n_clicks),
            unit="s"
        )
    )

    changed_click_rows.extend(selected_rows.tolist())

    click_plan_rows.append({
        "account_id": account_id,
        "burst_start": burst_start,
        "burst_seconds": burst_seconds,
        "modified_clicks": n_clicks
    })

click_plan = pd.DataFrame(click_plan_rows)

assert len(events) == TARGET_EVENTS

print("Planted clicking accounts:", len(click_ids))
print("Events modified:", len(changed_click_rows))
print("Total events:", len(events))

display(click_plan.head())

Planted clicking accounts: 300
Events modified: 2609
Total events: 300000


,account_id,burst_start,burst_seconds,modified_clicks
0,66,2026-09-08 03:21:59+00:00,10,12
1,100,2026-09-25 04:18:43+00:00,10,5
2,204,2026-09-27 22:57:57+00:00,10,4
3,290,2026-09-09 00:10:14+00:00,10,5
4,313,2026-09-11 19:39:33+00:00,120,11


In [16]:
click_check = events.loc[changed_click_rows]

assert click_check.index.is_unique
assert set(click_check["account_id"]) == set(click_ids)
assert click_check["event_type"].eq("click").all()
assert len(click_check) == click_plan["modified_clicks"].sum()

for plan in click_plan.itertuples(index=False):
    account_clicks = click_check[
        click_check["account_id"] == plan.account_id
    ]

    burst_end = plan.burst_start + pd.Timedelta(
        seconds=plan.burst_seconds
    )

    assert len(account_clicks) == plan.modified_clicks
    assert account_clicks["event_time"].ge(plan.burst_start).all()
    assert account_clicks["event_time"].lt(burst_end).all()

print("Verified clicking accounts:", len(click_plan))

Verified clicking accounts: 300


### Legitimate lookalike IDs

In [17]:
legitimate_ids = ground_truth.loc[
    ground_truth["pattern"] == "legitimate",
    "account_id"
].to_numpy()

lookalike_ids = rng.choice(
    legitimate_ids,
    size=800,
    replace=False
)

legit_shared_ids = lookalike_ids[:400]
legit_travel_ids = lookalike_ids[400:600]
legit_click_ids = lookalike_ids[600:]

assert len(np.unique(lookalike_ids)) == 800
assert set(lookalike_ids).issubset(set(legitimate_ids))

print("Shared-network accounts:", len(legit_shared_ids))
print("Travel accounts:", len(legit_travel_ids))
print("Genuine-click accounts:", len(legit_click_ids))

Shared-network accounts: 400
Travel accounts: 200
Genuine-click accounts: 200


In [18]:
legit_shared_groups = np.array_split(legit_shared_ids, 20)

block_numbers = rng.permutation(np.arange(1, 21))

changed_legit_shared_rows = []

for group, block_number in zip(
    legit_shared_groups,
    block_numbers
):
    start_offset = int(
        rng.integers(0, seconds_in_period - 600)
    )
    group_start = START + pd.Timedelta(seconds=start_offset)

    for account_id in group:
        account_rows = account_event_rows[account_id]
        n_posts = int(rng.integers(1, 6))

        selected_rows = rng.choice(
            account_rows,
            size=n_posts,
            replace=False
        )

        events.loc[selected_rows, "event_type"] = "post"

        events.loc[selected_rows, "event_time"] = (
            group_start
            + pd.to_timedelta(
                rng.integers(0, 600, size=n_posts),
                unit="s"
            )
        )

        last_octet = int(rng.integers(1, 255))

        events.loc[selected_rows, "ip_address"] = (
            f"10.250.{block_number}.{last_octet}"
        )

        changed_legit_shared_rows.extend(selected_rows.tolist())

assert len(events) == TARGET_EVENTS

print(
    "Legitimate shared-network accounts:",
    events.loc[changed_legit_shared_rows, "account_id"].nunique()
)
print("Events modified:", len(changed_legit_shared_rows))
print("Total events:", len(events))

display(events.loc[changed_legit_shared_rows[:5]])

Legitimate shared-network accounts: 400
Events modified: 1184
Total events: 300000


,account_id,event_time,event_type,country,ip_address
295061,11048,2026-09-21 02:31:22+00:00,post,IN,10.250.7.35
111187,11119,2026-09-21 02:35:47+00:00,post,DE,10.250.7.13
111181,11119,2026-09-21 02:29:01+00:00,post,DE,10.250.7.13
111180,11119,2026-09-21 02:31:53+00:00,post,DE,10.250.7.13
5085,509,2026-09-21 02:27:36+00:00,post,GB,10.250.7.51


In [19]:
busy_travel_ids = set(
    rng.choice(legit_travel_ids, size=50, replace=False)
)

legit_travel_plan_rows = []

for account_id in legit_travel_ids:
    home_country = country_lookup.loc[account_id]

    other_countries = [
        country for country in countries
        if country != home_country
    ]

    travel_offset = int(
        rng.integers(7 * 24 * 60 * 60, 20 * 24 * 60 * 60)
    )
    travel_time = START + pd.Timedelta(seconds=travel_offset)

    busy_visit = account_id in busy_travel_ids

    activity_window = (
        1800 if busy_visit
        else int((END - travel_time).total_seconds())
    )

    legit_travel_plan_rows.append({
        "account_id": account_id,
        "home_country": home_country,
        "new_country": rng.choice(other_countries),
        "travel_time": travel_time,
        "activity_window_seconds": activity_window,
        "busy_visit": busy_visit
    })

legit_travel_plan = pd.DataFrame(legit_travel_plan_rows)

print("Legitimate travelers:", len(legit_travel_plan))
print("Busy travelers:", legit_travel_plan["busy_visit"].sum())

display(legit_travel_plan.head())

Legitimate travelers: 200
Busy travelers: 50


,account_id,home_country,new_country,travel_time,activity_window_seconds,busy_visit
0,1900,US,DE,2026-09-10 22:04:07+00:00,1800,True
1,16643,DE,GB,2026-09-13 05:22:27+00:00,1800,True
2,18800,GB,CA,2026-09-18 04:02:58+00:00,1108622,False
3,1975,GB,IN,2026-09-19 11:48:32+00:00,1800,True
4,8295,US,GB,2026-09-20 23:15:45+00:00,866655,False


In [20]:
changed_legit_travel_rows = []
legit_travel_history_rows = []

for plan in legit_travel_plan.itertuples(index=False):
    account_id = plan.account_id

    account_rows = rng.permutation(
        account_event_rows[account_id]
    )

    history_rows = account_rows[:6]
    travel_rows = account_rows[6:]

    events.loc[history_rows, "event_time"] = (
        START
        + pd.to_timedelta(
            rng.integers(0, 7 * 24 * 60 * 60, size=6),
            unit="s"
        )
    )

    events.loc[history_rows, "country"] = plan.home_country
    events.loc[history_rows, "ip_address"] = ip_lookup[account_id]
    events.loc[history_rows[0], "event_type"] = "login"

    travel_offsets = rng.integers(
        1,
        plan.activity_window_seconds,
        size=len(travel_rows)
    )
    travel_offsets[0] = 0

    events.loc[travel_rows, "event_time"] = (
        plan.travel_time
        + pd.to_timedelta(travel_offsets, unit="s")
    )

    events.loc[travel_rows, "country"] = plan.new_country

    events.loc[travel_rows, "event_type"] = rng.choice(
        ["post", "click"],
        size=len(travel_rows)
    )
    events.loc[travel_rows[0], "event_type"] = "login"

    a, b, c = rng.integers(1, 255, size=3)
    events.loc[travel_rows, "ip_address"] = f"10.{a}.{b}.{c}"

    legit_travel_history_rows.extend(history_rows.tolist())
    changed_legit_travel_rows.extend(travel_rows.tolist())

assert len(events) == TARGET_EVENTS

print(
    "Legitimate travelers modified:",
    events.loc[changed_legit_travel_rows, "account_id"].nunique()
)
print("Total events:", len(events))

example_id = legit_travel_ids[0]

display(
    events.loc[events["account_id"] == example_id]
    .sort_values("event_time")
)

Legitimate travelers modified: 200
Total events: 300000


,account_id,event_time,event_type,country,ip_address
207655,1900,2026-09-01 21:58:09+00:00,login,US,10.135.44.183
18999,1900,2026-09-02 00:17:03+00:00,click,US,10.135.44.183
18993,1900,2026-09-02 15:10:14+00:00,click,US,10.135.44.183
18992,1900,2026-09-04 12:10:55+00:00,login,US,10.135.44.183
18997,1900,2026-09-04 12:50:48+00:00,login,US,10.135.44.183
18990,1900,2026-09-05 01:38:13+00:00,login,US,10.135.44.183
18996,1900,2026-09-10 22:04:07+00:00,login,DE,10.170.236.107
18995,1900,2026-09-10 22:10:12+00:00,post,DE,10.170.236.107
18994,1900,2026-09-10 22:22:44+00:00,click,DE,10.170.236.107
237100,1900,2026-09-10 22:22:57+00:00,post,DE,10.170.236.107


In [21]:
changed_legit_click_rows = []
legit_click_plan_rows = []

for account_id in legit_click_ids:
    account_rows = account_event_rows[account_id]

    max_clicks = min(14, len(account_rows))
    n_clicks = int(rng.integers(4, max_clicks + 1))

    burst_seconds = int(rng.choice([10, 30, 120]))

    start_offset = int(
        rng.integers(0, seconds_in_period - burst_seconds)
    )
    burst_start = START + pd.Timedelta(seconds=start_offset)

    selected_rows = rng.choice(
        account_rows,
        size=n_clicks,
        replace=False
    )

    events.loc[selected_rows, "event_type"] = "click"

    events.loc[selected_rows, "event_time"] = (
        burst_start
        + pd.to_timedelta(
            rng.integers(0, burst_seconds, size=n_clicks),
            unit="s"
        )
    )

    changed_legit_click_rows.extend(selected_rows.tolist())

    legit_click_plan_rows.append({
        "account_id": account_id,
        "burst_start": burst_start,
        "burst_seconds": burst_seconds,
        "modified_clicks": n_clicks
    })

legit_click_plan = pd.DataFrame(legit_click_plan_rows)

assert len(events) == TARGET_EVENTS

print(
    "Legitimate clicking accounts:",
    events.loc[changed_legit_click_rows, "account_id"].nunique()
)
print("Events modified:", len(changed_legit_click_rows))
print("Total events:", len(events))

display(legit_click_plan.head())

Legitimate clicking accounts: 200
Events modified: 1756
Total events: 300000


,account_id,burst_start,burst_seconds,modified_clicks
0,17472,2026-09-07 17:50:37+00:00,120,8
1,1263,2026-09-09 23:48:54+00:00,120,4
2,7660,2026-09-24 10:49:01+00:00,30,10
3,8910,2026-09-08 05:34:28+00:00,10,11
4,12474,2026-09-02 04:55:32+00:00,10,13


In [22]:
assert len(accounts) == N_ACCOUNTS
assert len(events) == TARGET_EVENTS

assert accounts["account_id"].is_unique
assert ground_truth["account_id"].is_unique

assert set(events["account_id"]) == set(accounts["account_id"])
assert set(ground_truth["account_id"]) == set(accounts["account_id"])

assert accounts.notna().all().all()
assert events.notna().all().all()
assert ground_truth.notna().all().all()

assert events["event_type"].isin(["login", "post", "click"]).all()
assert events["country"].isin(countries).all()

assert events["event_time"].ge(START).all()
assert events["event_time"].lt(END).all()

signup_lookup = pd.to_datetime(
    accounts.set_index("account_id")["signup_date"],
    utc=True
)

event_signup_dates = events["account_id"].map(signup_lookup)

assert events["event_time"].ge(event_signup_dates).all()

assert events.groupby("account_id").size().ge(10).all()

assert ground_truth["pattern"].value_counts().to_dict() == PATTERN_COUNTS
assert "pattern" not in events.columns

print("Structural checks passed.")
print("Accounts:", len(accounts))
print("Events:", len(events))
print("Ground-truth rows:", len(ground_truth))

Structural checks passed.
Accounts: 20000
Events: 300000
Ground-truth rows: 20000


In [23]:
# Shared-network lookalikes
shared_check = events.loc[changed_legit_shared_rows]

assert shared_check.index.is_unique
assert set(shared_check["account_id"]) == set(legit_shared_ids)
assert shared_check["event_type"].eq("post").all()

for group in legit_shared_groups:
    posts = shared_check[
        shared_check["account_id"].isin(group)
    ]

    span = (
        posts["event_time"].max()
        - posts["event_time"].min()
    ).total_seconds()

    blocks = posts["ip_address"].str.rsplit(".", n=1).str[0]

    assert span < 600
    assert blocks.nunique() == 1

print("Shared-network accounts verified:", len(legit_shared_ids))


# Travel lookalikes
for plan in legit_travel_plan.itertuples(index=False):
    activity = events.loc[account_event_rows[plan.account_id]]

    history = activity[activity["event_time"] < plan.travel_time]
    travel = activity[activity["event_time"] >= plan.travel_time]

    assert len(history) == 6
    assert history["country"].eq(plan.home_country).all()
    assert history["event_type"].eq("login").any()
    assert travel["country"].eq(plan.new_country).all()

    window_end = plan.travel_time + pd.Timedelta(
        seconds=plan.activity_window_seconds
    )
    assert travel["event_time"].lt(window_end).all()

    first = travel.sort_values("event_time").iloc[0]
    assert first["event_time"] == plan.travel_time
    assert first["event_type"] == "login"

print("Travel accounts verified:", len(legit_travel_plan))


# Genuine click spikes
click_check = events.loc[changed_legit_click_rows]

assert click_check.index.is_unique
assert set(click_check["account_id"]) == set(legit_click_ids)
assert click_check["event_type"].eq("click").all()

for plan in legit_click_plan.itertuples(index=False):
    clicks = click_check[
        click_check["account_id"] == plan.account_id
    ]

    window_end = plan.burst_start + pd.Timedelta(
        seconds=plan.burst_seconds
    )

    assert len(clicks) == plan.modified_clicks
    assert clicks["event_time"].ge(plan.burst_start).all()
    assert clicks["event_time"].lt(window_end).all()

print("Genuine-click accounts verified:", len(legit_click_plan))


# Labels remain legitimate
lookalike_labels = ground_truth.set_index("account_id").loc[
    lookalike_ids, "pattern"
]

assert lookalike_labels.eq("legitimate").all()

print("All lookalike checks passed.")

Shared-network accounts verified: 400
Travel accounts verified: 200
Genuine-click accounts verified: 200
All lookalike checks passed.


### Assigning Event IDs

In [24]:
events = events.sort_values(
    "event_time",
    kind="stable"
).reset_index(drop=True)

events["event_id"] = np.arange(1, len(events) + 1)

events = events[
    [
        "event_id",
        "account_id",
        "event_time",
        "event_type",
        "ip_address",
        "country"
    ]
]

assert events["event_id"].is_unique
assert events["event_time"].is_monotonic_increasing

accounts.to_csv(DATA_DIR / "accounts.csv", index=False)

events.to_csv(
    DATA_DIR / "events.csv",
    index=False,
    date_format="%Y-%m-%d %H:%M:%S"
)

ground_truth.to_csv(
    DATA_DIR / "ground_truth.csv",
    index=False
)

print("Saved to:", DATA_DIR)
print("Accounts:", len(accounts))
print("Events:", len(events))
print("Ground-truth rows:", len(ground_truth))

events.head()

Saved to: /content/drive/MyDrive/Project: trust-safety/02-abuse-detection-sql/data
Accounts: 20000
Events: 300000
Ground-truth rows: 20000


,event_id,account_id,event_time,event_type,ip_address,country
0,1,7651,2026-09-01 00:00:03+00:00,click,10.53.89.192,DE
1,2,5805,2026-09-01 00:00:05+00:00,click,10.85.62.234,GB
2,3,11491,2026-09-01 00:00:20+00:00,click,10.169.161.2,DE
3,4,18929,2026-09-01 00:00:27+00:00,click,10.240.241.64,US
4,5,12181,2026-09-01 00:00:38+00:00,click,10.37.101.144,DE


## Synthetic dataset summary

### Summary

Generated using seed 42 for September 1–30, 2026.
All event timestamps are UTC; October 1 is excluded.

### Tables
- accounts.csv: 20000 accounts.
- events.csv: 300000 events with unique, chronological event IDs.
- ground_truth.csv: one pattern label per account.

Ground-truth labels and generator plans are excluded from
SQL detection inputs.

### Planted accounts
- 19000 legitimate accounts.
- 400 spam accounts in 20 coordinated groups.
- 300 takeover accounts.
- 300 automated-clicking accounts.

Each planted abusive account belongs to one pattern.

### Legitimate lookalikes
Within the legitimate population, disjoint groups include:
- 400 accounts with coordinated shared-network posting.
- 200 travelers, including 50 with concentrated activity.
- 200 accounts with genuine click spikes.

Their labels remain legitimate. Some observable behaviour deliberately overlaps with planted abuse.

### Generator assumptions
Every account initially received ten events, with remaining
events randomly assigned. Baseline event-type probabilities
were 30% login, 20% post, and 50% click.

These probabilities, burst windows, and pattern counts are
design choices, not estimates of real platform behaviour.
Final event-type proportions differ after pattern injection.

### Validation
Checks passed for table sizes, account references, required
values, allowed event types, timestamps, signup ordering,
label counts, and generated pattern timelines.

These checks establish construction consistency, not realism
or detection effectiveness. Rule evaluation remains in-sample,
with no held-out dataset.

### Saved data
The three CSV files are stored in Google Drive under:
Project: trust-safety/02-abuse-detection-sql/data

SQL detection and evaluation have not yet been performed.